In [2]:
import pandas as pd
df_cash =pd.read_csv("C:/Users/admin/Downloads/datasets/daily_cashflow_dataset.csv")

In [3]:
df_cash.info()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype
---  ------           --------------   -----
 0   date             300000 non-null  str  
 1   inflow_type      300000 non-null  str  
 2   outflow_type     300000 non-null  str  
 3   expected_amount  300000 non-null  int64
 4   actual_amount    300000 non-null  int64
 5   variance         300000 non-null  int64
dtypes: int64(3), str(3)
memory usage: 23.4 MB


In [4]:
df_cash.head()

,date,inflow_type,outflow_type,expected_amount,actual_amount,variance
0,2023-07-09,Fees,Interest Payment,4174650,3827391,-347259
1,2022-01-10,EMI,Operational Expense,4130107,4793421,663314
2,2024-09-30,Fees,Principal Repayment,1513713,4631343,3117630
3,2023-10-07,Fees,Operational Expense,4968504,1084464,-3884040
4,2024-05-25,Fees,Interest Payment,2750623,811231,-1939392


In [5]:

df_cash["variance"] = df_cash["actual_amount"] - df_cash["expected_amount"]
df_cash["Abs_variance"] = df_cash["variance"].abs()

In [6]:
df_cash["Abs_variance"]

0          347259
1          663314
2         3117630
3         3884040
4         1939392
           ...   
299995     164439
299996    2238108
299997     217763
299998    2647701
299999    2353531
Name: Abs_variance, Length: 300000, dtype: int64

In [8]:
import numpy as np
threshold_high = df_cash["Abs_variance"].quantile(0.75)

df_cash["high_variance_flag"] = df_cash["Abs_variance"] > threshold_high

df_cash["high_variance_insight"] = np.where(
    df_cash["high_variance_flag"],
    "Forecasting issue / Behavioral uncertainty",
    ""
)

In [9]:
df_cash["high_variance_flag"]

0         False
1         False
2          True
3          True
4         False
          ...  
299995    False
299996    False
299997    False
299998     True
299999    False
Name: high_variance_flag, Length: 300000, dtype: bool

In [10]:

df_cash["high_variance_insight"]

0                                                   
1                                                   
2         Forecasting issue / Behavioral uncertainty
3         Forecasting issue / Behavioral uncertainty
4                                                   
                             ...                    
299995                                              
299996                                              
299997                                              
299998    Forecasting issue / Behavioral uncertainty
299999                                              
Name: high_variance_insight, Length: 300000, dtype: str

In [11]:

df_cash["neg_var_streak"] = (
    df_cash["variance"] < 0
).rolling(window=3).sum()

df_cash["consistent_negative_flag"] = df_cash["neg_var_streak"] == 3

df_cash["liquidity_risk_insight"] = np.where(
    df_cash["consistent_negative_flag"],
    "Overestimation of inflows → Liquidity risk",
    ""
)

In [12]:
df_cash["neg_var_streak"]

0         NaN
1         NaN
2         1.0
3         1.0
4         2.0
         ... 
299995    2.0
299996    3.0
299997    3.0
299998    2.0
299999    2.0
Name: neg_var_streak, Length: 300000, dtype: float64

In [13]:
df_cash["liquidity_risk_insight"]

0                                                   
1                                                   
2                                                   
3                                                   
4                                                   
                             ...                    
299995                                              
299996    Overestimation of inflows → Liquidity risk
299997    Overestimation of inflows → Liquidity risk
299998                                              
299999                                              
Name: liquidity_risk_insight, Length: 300000, dtype: str

In [14]:

df_cash["rolling_volatility"] = df_cash["variance"].rolling(window=5).std()

vol_threshold = df_cash["rolling_volatility"].quantile(0.75)

df_cash["high_volatility_flag"] = df_cash["rolling_volatility"] > vol_threshold

df_cash["volatility_insight"] = np.where(
    df_cash["high_volatility_flag"],
    "Unstable liquidity → Use conservative planning",
    ""
)

In [15]:
df_cash["volatility_insight"] 

0                                                       
1                                                       
2                                                       
3                                                       
4         Unstable liquidity → Use conservative planning
                               ...                      
299995                                                  
299996                                                  
299997                                                  
299998                                                  
299999                                                  
Name: volatility_insight, Length: 300000, dtype: str

In [17]:
df_cash["date"] = pd.to_datetime(df_cash["date"], errors="coerce")

In [18]:

df_cash["day"] = df_cash["date"].dt.day
df_cash["weekday"] = df_cash["date"].dt.weekday


spike_threshold = df_cash["Abs_variance"].quantile(0.90)

df_cash["spike_flag"] = df_cash["Abs_variance"] > spike_threshold

df_cash["seasonal_spike_insight"] = np.where(
    df_cash["spike_flag"],
    "Potential seasonal spike → Model in forecast",
    ""
)

In [19]:
df_cash["seasonal_spike_insight"]

0                                                     
1                                                     
2                                                     
3         Potential seasonal spike → Model in forecast
4                                                     
                              ...                     
299995                                                
299996                                                
299997                                                
299998                                                
299999                                                
Name: seasonal_spike_insight, Length: 300000, dtype: str